# R2000: paper Table 5, five independent EMA + DySample models

Three partitions train, one validates, and one tests (**3:1:1**).
Exact partition counts (images/instances): 405/456, 405/460, 405/458,
405/447, 404/460. All 2024 images and 2281 instances are retained.
The paper does not publish image memberships; these are reconstructed
assignments matching its counts, with exact duplicates kept together.
All 16 classes must appear in each train/validation/test subset.

Corrected RGG + EMA backbone and DySample neck; nano, `yolov8n.pt`,
300 epochs per fold, AdamW 0.001, zero bias warmup, batch 32 across
both T4 GPUs (16 each). Five models initialize independently.
Each best checkpoint is selected using validation, then tested.
Results are averaged across folds. This notebook executes all five
independent folds sequentially in one GPU session.

In [ ]:
import base64, io, os, sys, zipfile
from pathlib import Path
PROJECT = Path('/kaggle/working/paddy_yolo_paper_source')
PROJECT.mkdir(parents=True, exist_ok=True)
B64_SOURCE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_SOURCE))) as archive:
    archive.extractall(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/paper_settings/ultralytics'
os.environ['MPLCONFIGDIR'] = '/kaggle/working/paper_settings/matplotlib'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
import json
RUN = json.loads((PROJECT / 'kaggle/ema_dysample/paper_5fold/run_config.json').read_text())
print(json.dumps(RUN, indent=2))
sys.path.insert(0, str(PROJECT / 'kaggle/ema_dysample'))
from runtime import configure_ultralytics, run_logged
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r',
            str(PROJECT / 'kaggle/ema_dysample/paper_5fold/requirements.txt')],
           Path('/kaggle/working/environment_setup.log'), PROJECT)
import torch, ultralytics
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2
assert ultralytics.__version__ == '8.3.18'
configure_ultralytics()
run_logged([sys.executable, '-m', 'pytest', '-q', '--tb=short'],
           Path('/kaggle/working/preflight_tests.log'), PROJECT)

In [ ]:
sys.path.insert(0, str(PROJECT / 'kaggle/baseline'))
from prepare_dataset import prepare_dataset
from paddy_yolo.dataset.paper_folds import prepare_paper_folds
STAGED = prepare_dataset(Path('/kaggle/input'), Path('/kaggle/working/paper_original_split'))
DATA = Path('/kaggle/working/paper_5fold_splits')
prepare_paper_folds(STAGED, DATA, seed=RUN['seed'])
manifest = json.loads((DATA / 'fold_manifest.json').read_text())
print(json.dumps(manifest['partition_counts'], indent=2))
from ultralytics import YOLO
pretrained = YOLO(RUN['initial_weights'])
del pretrained

In [ ]:
OUTPUT = Path('/kaggle/working/ema_dysample_paper_5fold_output')
run_logged([sys.executable, str(PROJECT / 'kaggle/ema_dysample/paper_5fold/run_folds.py'),
            '--data-dir', str(DATA), '--output-dir', str(OUTPUT)],
           Path('/kaggle/working/paper_training.log'), PROJECT)
print((OUTPUT / 'summary.json').read_text())